## 数据入库

以 markdown # 拆分

In [5]:
file_path = '../api/mfd.md'

# 段落拆分
text_lines = []
with open(file_path, "r") as file:
    file_text = file.read()
    text_lines = file_text.split("# ")
print(text_lines[:3])

['#', '中华人民共和国民法典\n\n##', '（二）物权编\n\n###']


### Embedding入库

In [9]:
from pymilvus import model as milvus_model, MilvusClient
from tqdm import tqdm

# 初始化库
embedding_model = milvus_model.DefaultEmbeddingFunction()
# 数据库文件
milvus_client = MilvusClient(uri="./milvus_mfd.db")
# 数据库
collection_name = "mfd_rag_collection"
# 确定向量维度
test_embedding = embedding_model.encode_queries(["test"])[0]
embedding_dim = len(test_embedding)

# demo 测试，如果存在collection，先删除
if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)
milvus_client.create_collection(
    collection_name=collection_name,
    dimension=embedding_dim, # 向量维度
    metric_type="IP",  # 内积距离
    consistency_level="Strong",  # 支持的值为 (`"Strong"`, `"Session"`, `"Bounded"`, `"Eventually"`)。更多详情请参见 https://milvus.io/docs/consistency.md#Consistency-Level。
)

# 数据
data = []
# 向量化
doc_embeddings = embedding_model.encode_documents(text_lines)

for i, line in enumerate(tqdm(text_lines, desc="创建向量")):
    data.append({"id": i, "vector": doc_embeddings[i], "text": line})
milvus_client.insert(collection_name=collection_name, data=data)


创建向量: 100%|██████████| 30/30 [00:00<00:00, 252668.92it/s]


{'insert_count': 30, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29], 'cost': 0}

### 构建RAG

In [10]:
# 指定问题
question = "有人偷了我的东西怎么办？"

search_res = milvus_client.search(
    collection_name=collection_name,
    data=embedding_model.encode_queries(
        [question]
    ),  # 将问题转换为嵌入向量
    limit=3,  # 返回前3个结果
    search_params={"metric_type": "IP", "params": {}},  # 内积距离
    output_fields=["text"],  # 返回 text 字段
)
# 打印第一个看一下
print(search_res[0][0])

{'id': 21, 'distance': 0.614084005355835, 'entity': {'text': '二、权利质权\n\n**第四百四十九条** 可以出质的权利包括：\n（一）汇票、本票、支票；\n（二）债券、存款单；\n（三）仓单、提单；\n（四）可以转让的基金份额、股权；\n（五）可以转让的注册商标专用权、专利权、著作权等知识产权中的财产权；\n（六）应收账款；\n（七）法律、行政法规规定可以出质的其他财产权利。\n\n**第四百五十条** 以汇票、本票、支票、债券、存款单、仓单、提单出质的，当事人应当订立书面合同。质权自权利凭证交付之日起设立。\n\n**第四百五十一条** 以记名股票出质的，当事人应当订立书面合同。质权自股票交付之日起设立。\n以未上市公司股权出质的，适用公司法有关股权转让的规定。\n\n**第四百五十二条** 以可以转让的基金份额、股权出质的，当事人应当订立书面合同。质权自基金份额、股权登记于证券登记结算机构或者公司章程载明的股权登记簿时设立。\n以未上市公司股权出质的，适用公司法有关股权转让的规定。\n\n**第四百五十三条** 以可以转让的注册商标专用权、专利权、著作权等知识产权中的财产权出质的，当事人应当订立书面合同。质权自权利质押登记于相关部门时设立。\n\n**第四百五十四条** 以应收账款出质的，当事人应当订立书面合同。质权自应收账款质押登记于中国人民银行征信中心时设立。\n\n**第四百五十五条** 以法律、行政法规规定可以出质的其他财产权利出质的，依照法律、行政法规的规定。\n\n**第四百五十六条** 权利质权除适用本节规定外，参照适用本章动产质权的有关规定。\n\n####'}}


In [14]:
# retrieved_lines_with_distances = [
#     (res["entity"]["text"], res["distance"]) for res in search_res[0]
# ]
context = "".join(
    [res["entity"]["text"] for res in search_res[0]]
)
print(f"{context[:100]}.......")

二、权利质权

**第四百四十九条** 可以出质的权利包括：
（一）汇票、本票、支票；
（二）债券、存款单；
（三）仓单、提单；
（四）可以转让的基金份额、股权；
（五）可以转让的注册商标专用权、专利.......


In [15]:
# 组装PROMPT
SYSTEM_PROMPT = """
Human: 你是一个 AI 助手。你能够从提供的上下文段落片段中找到问题的答案。
"""

USER_PROMPT = f"""
请使用以下用 <context> 标签括起来的信息片段来回答用 <question> 标签括起来的问题。最后追加原始回答的中文翻译，并用 <translated>和</translated> 标签标注。
<context>
{context}
</context>
<question>
{question}
</question>
<translated>
</translated>
"""

# 初始化大模型
import os
from openai import OpenAI

api_key = os.getenv("DEEPSEEK_API_KEY")
deepseek_client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/v1"
)

response = deepseek_client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
)
print(response.choices[0].message.content)

根据提供的上下文信息，没有直接涉及盗窃问题的处理规定。上下文主要涵盖了权利质权、合同变更和转让以及地役权等民事法律条款，并未提及盗窃等刑事或侵权行为的应对措施。

<translated>
根据提供的上下文信息，没有直接涉及盗窃问题的处理规定。上下文主要涵盖了权利质权、合同变更和转让以及地役权等民事法律条款，并未提及盗窃等刑事或侵权行为的应对措施。
</translated>
